# Đặc trưng bộ dữ liệu giấy tờ

Mô tả golden set do `src/data-golden-set-generator` sinh ra theo cách các bài báo về bộ dữ liệu OCR/Document AI (FUNSD, CORD, SROIE, DocBank, RVL-CDIP) thường trình bày. Mỗi phần trả lời một câu hỏi:

| Phần | Câu hỏi |
|---|---|
| 1. Quy mô | Mỗi loại giấy tờ có bao nhiêu hồ sơ, trang, ảnh, trường, dòng chữ? Có cân bằng không? |
| 2. Phân bố lớp | Số mẫu theo loại, theo biến thể mẫu và theo thời kỳ ban hành |
| 3. Ma trận thuộc tính | Loại nào khó ở trục nào: viết tay, đa trang, sơ đồ, mẫu thay đổi, giấy cũ |
| 4. Ảnh mẫu | Mỗi loại trông thế nào, trường nằm ở đâu, nhiễu làm ảnh thay đổi ra sao |
| 5. Đặc trưng ảnh, văn bản, bố cục | Độ nghiêng, độ tương phản, độ nét, nhiễu; số ký tự, tỷ lệ dấu; mật độ trường |
| 6. Lược đồ nhãn | `ground_truth.json` có những cấp nhãn nào |
| 7. Không gian đặc trưng | Loại nào tách rời, loại nào chồng lấn khi nhìn bằng embedding ảnh |
| 8. Benchmark theo loại | Đặc trưng khác nhau có thật sự làm OCR khó hơn không (lấy kết quả từ `ocr-eval.ipynb`) |

Phần 1, 3, 8 là tối thiểu để người đọc hiểu bộ dữ liệu và thấy vì sao phải chia theo loại giấy tờ. Hình và bảng được lưu vào `outputs/dataset-profile/` để chèn vào báo cáo.

Dữ liệu là hồ sơ giả lập (ghi "MẪU GIẢ LẬP"), không có thông tin cá nhân thật nên ảnh mẫu không cần che. Khi chạy notebook trên `real/` thì phải che trước khi xuất hình.

## 0. Cấu hình

Dùng `jupyters/.venv` như `ocr-eval.ipynb` (xem `README.md`). Phần 7 cần `torch`, `torchvision`, `scikit-learn`; phần 8 cần đã chạy `ocr-eval.ipynb`. Các phần khác chỉ đọc `ground_truth.json` và ảnh.

In [ ]:
import json, re, unicodedata
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

KAGGLE_DATA = Path("/kaggle/input/datasets/balebomm/dataset/generated")  # dataset Kaggle, chỉ đọc
DATA_DIR = KAGGLE_DATA if KAGGLE_DATA.is_dir() else Path("data")  # thư mục generated/ của generator (Colab: /content/data)
OUTPUT_DIR = Path("outputs")             # ocr-eval.ipynb ghi scores_e2e.csv, scores_rec.csv ở đây
FIG_DIR = OUTPUT_DIR / "dataset-profile" # hình PNG, bảng CSV, cache đo ảnh và embedding
TEMPLATES_DIR = Path("../src/data-golden-set-generator/templates")  # schema.yaml: nhãn tiếng Việt của trường. Không có thì bỏ qua
LEVELS = ["clean", "light", "medium", "heavy"]

# Thứ tự và tên ngắn trên biểu đồ. Loại chưa khai báo ở đây vẫn được vẽ, tên là doc_type
DOC_TYPES = {
    "can-cuoc": "Căn cước",
    "giay-chung-tu": "Giấy chứng tử",
    "giay-chung-nhan-nha-dat": "GCN nhà đất",
    "ban-ve-hien-trang": "Bản vẽ hiện trạng",
    "to-dang-ky-nha-dat": "Tờ đăng ký nhà đất",
    "so-ho-khau": "Sổ hộ khẩu",
    "giay-sang-dat": "Giấy sang đất",
}

# Thuộc tính bố cục không có trong ground_truth.json, khai báo tay theo template (0 = không, 1 = có).
# Thêm loại giấy tờ mới thì thêm dòng ở đây
DECLARED_TRAITS = {
    #                          2 mặt  Bảng/sơ đồ  Mẫu in sẵn
    "can-cuoc":                (1,    0,          1),
    "giay-chung-tu":           (0,    0,          1),
    "giay-chung-nhan-nha-dat": (0,    1,          1),
    "ban-ve-hien-trang":       (0,    1,          1),
    "to-dang-ky-nha-dat":      (0,    0,          1),
    "so-ho-khau":              (0,    1,          1),
    "giay-sang-dat":           (0,    0,          0),
}
DECLARED_COLUMNS = ["2 mặt", "Bảng/sơ đồ", "Mẫu in sẵn"]

IMAGE_MAX_SIDE = 1024                    # thu nhỏ ảnh trước khi đo (phần 5): nhanh hơn, số đo so được giữa các ảnh
EMBED_MODEL = "resnet18"                 # backbone torchvision họ ResNet, trọng số ImageNet
EMBED_LEVELS = ["clean", "heavy"]        # mức nhiễu đưa vào phần 7
MIN_FIELDS = 20                          # phần 8: ô có ít trường hơn thì để trống

In [ ]:
# Màu: mỗi loại giấy tờ một màu cố định (thứ tự slot, đã kiểm tra CVD), mức nhiễu dùng dải xanh đậm dần
CATEGORICAL = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
MARKERS = ["o", "s", "^", "D", "v", "P", "X", "*"]
LEVEL_COLOR = dict(zip(LEVELS, ["#86b6ef", "#3987e5", "#1c5cab", "#0d366b"]))
FILL_COLOR = {"printed": "#a3aab2", "typed": "#2a78d6", "handwritten": "#eb6834"}
FILL_NAME = {"printed": "In sẵn", "typed": "Đánh máy", "handwritten": "Viết tay"}
SEQ = LinearSegmentedColormap.from_list("seq_blue", ["#cde2fb", "#86b6ef", "#3987e5", "#256abf", "#184f95", "#0d366b"])
INK, MUTED, GRID, SURFACE = "#1f2328", "#59636e", "#d8dee4", "#fcfcfb"
plt.rcParams.update({"figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
                     "axes.titlesize": 11, "axes.labelcolor": MUTED, "text.color": INK})
pct = plt.FuncFormatter(lambda v, _: f"{v * 100:.4g}%")

def style_axes(ax, grid="y", percent_axis=None):
    ax.grid(axis=grid, color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color(GRID)
    ax.tick_params(colors=MUTED, length=0)
    if percent_axis:
        getattr(ax, f"{percent_axis}axis").set_major_formatter(pct)

FIG_DIR.mkdir(parents=True, exist_ok=True)

def save(fig, name):
    fig.savefig(FIG_DIR / f"{name}.png", dpi=200, bbox_inches="tight")

def save_table(df, name):
    df.to_csv(FIG_DIR / f"{name}.csv", encoding="utf-8-sig")   # BOM để Excel đọc đúng tiếng Việt
    return df

## Đọc ground truth

Quét mọi `ground_truth.json` trong `DATA_DIR`, dựng 4 bảng:

- `dossiers`: mỗi hồ sơ một dòng (nhãn quyết định, số ca lỗi).
- `docs`: mỗi giấy tờ một dòng (loại, biến thể, ngày ban hành, số trang, thông số render: khổ giấy, độ cũ, nếp gấp, độ nghiêng).
- `fields`: mỗi trường một dòng (kiểu dữ liệu, kiểu điền, chuỗi in trên giấy, số dòng bbox).
- `pages`: mỗi ảnh trang một dòng, cả bản sạch lẫn bản nhiễu (phép biến đổi đã áp, văn bản trang).

In [ ]:
def op_value(ops, name):
    """'rotate:0.32' -> 0.32. Không có phép đó thì 0."""
    for op in ops:
        k, _, v = op.partition(":")
        if k == name:
            return float(v)
    return 0.0

manifests = {p.parent: json.loads(p.read_text(encoding="utf-8")) for p in DATA_DIR.rglob("manifest-*.json")}
dossier_rows, doc_rows, field_rows, page_rows = [], [], [], []
for gt_path in sorted(DATA_DIR.rglob("ground_truth.json")):
    gt = json.loads(gt_path.read_text(encoding="utf-8"))
    dpi = manifests.get(gt_path.parent.parent, {}).get("dpi")
    decision = gt.get("decision", {})
    dossier_rows.append(dict(
        dossier=gt["dossier_id"], procedure=gt["procedure"], label=decision.get("label"),
        rules=[r.get("rule") for r in decision.get("reasons", [])],
        missing=len(gt.get("missing_documents", [])), review=gt.get("review", {}).get("status"),
    ))
    for doc in gt["documents"]:
        r, texts = doc.get("render", {}), doc["files"].get("text", [])
        key = dict(dossier=gt["dossier_id"], doc_id=doc["doc_id"], doc_type=doc["doc_type"], variant=doc.get("variant") or "—")
        doc_rows.append(dict(
            **key, issue_date=pd.to_datetime(doc.get("issue_date")), n_pages=len(doc["files"]["pages"]),
            page_format=r.get("page"), paper=r.get("paper"), w_mm=r.get("w_mm"), h_mm=r.get("h_mm"),
            age=r.get("age", 0.0), folds=len(r.get("folds") or []), tilt=r.get("tilt", 0.0),
            print_font=r.get("print_font"), dpi=dpi,
            n_fields=len(doc["fields"]), n_mutated=sum(f.get("mutated_by") is not None for f in doc["fields"]),
        ))
        for f in doc["fields"]:
            field_rows.append(dict(
                **key, field=f["name"], field_type=f["type"], fill=f.get("fill") or "printed",
                text=f.get("text") or "", mutated=f.get("mutated_by") is not None, bbox=f.get("bbox") or [],
            ))
        noisy = [("clean", i + 1, rel, []) for i, rel in enumerate(doc["files"]["pages"])]
        noisy += [(n["level"], n["page"], n["path"], n.get("ops", [])) for n in doc["files"].get("noisy", [])]
        for level, page, rel, ops in noisy:
            page_rows.append(dict(
                **key, level=level, page=page, path=(gt_path.parent / rel).as_posix(), ops=ops,
                text=texts[page - 1] if page <= len(texts) else "",
                skew=r.get("tilt", 0.0) + op_value(ops, "rotate"),   # độ nghiêng thật, lấy từ generator thay vì ước lượng
                blur=op_value(ops, "blur"), jpeg=op_value(ops, "jpeg") or None,
            ))

if not doc_rows:
    raise FileNotFoundError(f"Không thấy ground_truth.json trong {DATA_DIR.resolve()}. Xem README.md mục Dữ liệu.")

dossiers, docs, fields, pages = map(pd.DataFrame, (dossier_rows, doc_rows, field_rows, page_rows))
docs[["w_mm", "dpi"]] = docs[["w_mm", "dpi"]].apply(pd.to_numeric)
pages = pages[pages.path.map(lambda p: Path(p).exists())].reset_index(drop=True)

# doc_type là categorical có thứ tự: bảng và biểu đồ luôn cùng thứ tự, màu đi theo loại
TYPE_ORDER = [t for t in DOC_TYPES if t in set(docs.doc_type)] + sorted(set(docs.doc_type) - set(DOC_TYPES))
NAME = {t: DOC_TYPES.get(t, t) for t in TYPE_ORDER}
COLOR = {t: CATEGORICAL[i % len(CATEGORICAL)] for i, t in enumerate(TYPE_ORDER)}
MARKER = {t: MARKERS[i % len(MARKERS)] for i, t in enumerate(TYPE_ORDER)}
for df in (docs, fields, pages):
    df["doc_type"] = pd.Categorical(df["doc_type"], TYPE_ORDER, ordered=True)
pages["level"] = pd.Categorical(pages["level"], LEVELS, ordered=True)
fields["chars"] = fields.text.str.len()
fields["n_lines"] = fields.bbox.map(len)

print(f"{len(dossiers)} hồ sơ, {len(docs)} giấy tờ, {len(TYPE_ORDER)} loại, "
      f"{(pages.level == 'clean').sum()} trang, {len(pages)} ảnh, {len(fields)} trường")

## 1. Bảng thống kê quy mô

Mỗi loại giấy tờ một dòng. **Dòng chữ** là số bbox có nhãn (mỗi dòng của một trường một bbox). **Ký tự** tính trên chuỗi in trên giấy của các trường có nhãn, không tính chữ in sẵn của mẫu.

Golden set chỉ để đánh giá nên không chia train/val/test. Nếu cần chia (ví dụ để tinh chỉnh model) thì chia theo **hồ sơ**, không chia theo trang: các giấy tờ trong một hồ sơ dùng chung người và thửa đất, chia theo trang sẽ rò thông tin giữa các tập.

In [ ]:
def by_type(df):
    return df.groupby("doc_type", observed=True)

clean = pages[pages.level == "clean"]
scale = pd.DataFrame({
    "Hồ sơ": by_type(docs).dossier.nunique(),
    "Giấy tờ": by_type(docs).size(),
    "Biến thể": by_type(docs).variant.nunique(),
    "Trang": by_type(clean).size(),
    "Trang/giấy tờ": by_type(docs).n_pages.mean(),
    "Ảnh (sạch + nhiễu)": by_type(pages).size(),
    "Trường": by_type(fields).size(),
    "Dòng chữ": by_type(fields).n_lines.sum(),
    "Ký tự": by_type(fields).chars.sum(),
    "Trường bị gây lỗi": by_type(fields).mutated.sum(),
})
scale.index = scale.index.astype(str)
total = scale.sum().rename("Tổng")
total["Hồ sơ"] = docs.dossier.nunique()
total["Trang/giấy tờ"] = docs.n_pages.mean()
scale = pd.concat([scale.rename(index=NAME), total.to_frame().T])
scale["% trang"] = scale["Trang"] / total["Trang"]
save_table(scale, "1-scale")
(scale.style
   .format("{:,.0f}").format("{:.1f}", subset=["Trang/giấy tờ"]).format("{:.1%}", subset=["% trang"])
   .bar(subset=pd.IndexSlice[scale.index[:-1], ["Trang", "Ký tự"]], color="#b7d3f6"))

Theo hồ sơ: nhãn quyết định (dùng cho đánh giá cuối, AIP-027) và trạng thái duyệt của chuyên viên.

In [ ]:
pd.concat({
    "Nhãn quyết định": dossiers.label.value_counts(),
    "Trạng thái duyệt": dossiers.review.value_counts(),
    "Quy tắc bị vi phạm": dossiers.rules.explode().dropna().value_counts(),
}).to_frame("Số hồ sơ")

## 2. Phân bố lớp

Trái: số trang mỗi loại (loại nhiều trang như sổ hộ khẩu chiếm phần lớn ảnh dù số giấy tờ bằng nhau). Phải: số giấy tờ theo biến thể mẫu. Generator xoay vòng kịch bản nên biến thể gần đều; lệch là do ràng buộc thực tế (ví dụ người trẻ không thể có CMND 9 số) hoặc do chạy `--coverage random`.

In [ ]:
def type_barh(ax, values, fmt="{:,.0f}"):
    y = np.arange(len(values))
    ax.barh(y, values.values, height=0.62, color=[COLOR[t] for t in values.index], edgecolor=SURFACE, linewidth=2)
    for yi, v in zip(y, values.values):
        ax.text(v, yi, " " + fmt.format(v), va="center", fontsize=9, color=INK)
    ax.set_yticks(y, [NAME[t] for t in values.index])
    ax.invert_yaxis()
    ax.set_xlim(0, values.max() * 1.2)
    style_axes(ax, grid="x")

multi = [t for t in TYPE_ORDER if docs[docs.doc_type == t].variant.nunique() > 1]
fig = plt.figure(figsize=(14, max(4, 0.9 * len(multi) + 1.5)))
gs = fig.add_gridspec(max(len(multi), 1), 2, width_ratios=[1.1, 1])
ax = fig.add_subplot(gs[:, 0])
type_barh(ax, by_type(clean).size())
ax.set_title("Số trang (ảnh sạch) theo loại", loc="left")
for i, t in enumerate(multi):
    ax = fig.add_subplot(gs[i, 1])
    counts = docs[docs.doc_type == t].variant.value_counts().sort_index()
    y = np.arange(len(counts))
    ax.barh(y, counts.values, height=0.62, color=COLOR[t], edgecolor=SURFACE, linewidth=2)
    for yi, v in zip(y, counts.values):
        ax.text(v, yi, f" {v}", va="center", fontsize=9)
    ax.set_yticks(y, counts.index, fontsize=9)
    ax.set_xlim(0, docs.groupby(["doc_type", "variant"], observed=True).size().max() * 1.2)
    ax.set_title(NAME[t], loc="left", fontsize=10)
    style_axes(ax, grid="x")
    ax.tick_params(labelbottom=(i == len(multi) - 1))
fig.suptitle("Phân bố lớp và biến thể mẫu", x=0.01, ha="left")
plt.tight_layout()
save(fig, "2-class-distribution")

**Thời kỳ ban hành.** Mỗi chấm là một giấy tờ, trục ngang là năm ban hành, màu là biến thể. Loại trải dài nhiều thời kỳ thì mẫu, font, cách ghi địa chỉ (3 cấp trước 01/7/2025, 2 cấp sau đó) thay đổi theo.

In [ ]:
rng = np.random.default_rng(0)
fig, ax = plt.subplots(figsize=(13, 0.55 * len(TYPE_ORDER) + 1.2))
for i, t in enumerate(TYPE_ORDER):
    d = docs[docs.doc_type == t].dropna(subset=["issue_date"])
    variants = sorted(d.variant.unique())
    for j, v in enumerate(variants):
        dv = d[d.variant == v]
        year = dv.issue_date.dt.year + dv.issue_date.dt.dayofyear / 366
        ax.scatter(year, i + rng.uniform(-0.22, 0.22, len(dv)), s=36, color=COLOR[t], marker=MARKERS[j % len(MARKERS)],
                   edgecolor=SURFACE, linewidth=1, alpha=0.9, label=f"{NAME[t]}: {v}" if len(variants) > 1 else None)
ax.set_yticks(range(len(TYPE_ORDER)), [NAME[t] for t in TYPE_ORDER])
ax.invert_yaxis()
ax.axvline(2025.5, color=MUTED, linewidth=1, linestyle="--")
ax.text(2025.5, -0.6, " 01/7/2025: địa chỉ 2 cấp", fontsize=8, color=MUTED, va="bottom")
ax.set_xlabel("Năm ban hành")
ax.set_title("Thời kỳ ban hành theo loại và biến thể (hình dạng = biến thể)", loc="left")
style_axes(ax, grid="x")
ax.legend(frameon=False, fontsize=8, loc="upper left", bbox_to_anchor=(1.01, 1))
plt.tight_layout()
save(fig, "2-issue-period")

## 3. Ma trận thuộc tính (loại giấy tờ × đặc điểm)

Tách mô tả bằng lời của từng loại thành các trục so sánh được. Trục **đo từ dữ liệu** tính từ `ground_truth.json`, nên đổi generator hay thêm dữ liệu là ma trận tự cập nhật. Trục **khai báo** lấy từ `DECLARED_TRAITS` vì ground truth không có nhãn vùng bảng/sơ đồ.

| Trục | Cách đo |
|---|---|
| Đa trang | Số trang trung bình mỗi giấy tờ |
| Viết tay / Đánh máy | Tỷ lệ ký tự có nhãn thuộc trường viết tay / đánh máy máy chữ |
| Biến thể mẫu | Số biến thể mẫu (mẫu đổi theo thời kỳ, thế hệ thẻ, kiểu điền) |
| Trải thời kỳ | Khoảng năm ban hành (năm lớn nhất − nhỏ nhất) |
| Giấy cũ | Độ cũ giấy trung bình (0 = mới, 1 = rất ố, mực phai) |
| Nếp gấp | Số nếp gấp trung bình mỗi giấy tờ |
| Mật độ trường | Số dòng chữ có nhãn mỗi trang |

Màu là giá trị chia cho giá trị lớn nhất của cột (0–1), chữ trong ô là giá trị thật. Bảng dạng chấm (●, ●●, ●●●) dùng mức 0–3 lấy từ màu, để dán vào báo cáo.

In [ ]:
chars_by_fill = fields.pivot_table(index="doc_type", columns="fill", values="chars", aggfunc="sum", observed=True).fillna(0)
share = chars_by_fill.div(chars_by_fill.sum(axis=1), axis=0)
year = docs.issue_date.dt.year
measured = pd.DataFrame({
    "Đa trang": by_type(docs).n_pages.mean(),
    "Viết tay": share.get("handwritten", 0.0),
    "Đánh máy": share.get("typed", 0.0),
    "Biến thể mẫu": by_type(docs).variant.nunique(),
    "Trải thời kỳ": year.groupby(docs.doc_type, observed=True).agg(lambda s: s.max() - s.min()),
    "Giấy cũ": by_type(docs).age.mean(),
    "Nếp gấp": by_type(docs).folds.mean(),
    "Mật độ trường": by_type(fields).n_lines.sum() / by_type(clean).size(),
}).reindex(TYPE_ORDER)
measured.index = measured.index.astype(str)
declared = pd.DataFrame.from_dict({t: DECLARED_TRAITS.get(t, (np.nan,) * len(DECLARED_COLUMNS)) for t in TYPE_ORDER},
                                  orient="index", columns=DECLARED_COLUMNS)
traits = measured.join(declared)
norm_traits = traits / traits.max().replace(0, 1)
save_table(traits.rename(index=NAME), "3-trait-matrix")

FMT = {"Viết tay": "{:.0%}", "Đánh máy": "{:.0%}", "Giấy cũ": "{:.2f}", "Nếp gấp": "{:.1f}", "Đa trang": "{:.1f}",
       "Mật độ trường": "{:.0f}", **{c: "{:.0f}" for c in ["Biến thể mẫu", "Trải thời kỳ", *DECLARED_COLUMNS]}}

fig, ax = plt.subplots(figsize=(1.15 * traits.shape[1] + 2.5, 0.55 * len(TYPE_ORDER) + 1.6))
ax.imshow(norm_traits.values, cmap=SEQ, vmin=0, vmax=1, aspect="auto")
for i, t in enumerate(TYPE_ORDER):
    for j, c in enumerate(traits.columns):
        v, n = traits.loc[t, c], norm_traits.loc[t, c]
        if pd.isna(v):
            continue
        text = ("●" if v else "") if c in DECLARED_COLUMNS else FMT[c].format(v)
        ax.text(j, i, text, ha="center", va="center", fontsize=9, color="white" if n > 0.55 else INK)
ax.axvline(len(measured.columns) - 0.5, color=SURFACE, linewidth=6)
ax.set_xticks(range(traits.shape[1]), traits.columns, rotation=30, ha="right")
ax.set_yticks(range(len(TYPE_ORDER)), [NAME[t] for t in TYPE_ORDER])
ax.tick_params(length=0, colors=INK)
for s in ax.spines.values():
    s.set_visible(False)
ax.text(len(measured.columns) / 2 - 0.5, -0.8, "đo từ dữ liệu", ha="center", color=MUTED, fontsize=9)
ax.text(len(measured.columns) + len(DECLARED_COLUMNS) / 2 - 0.5, -0.8, "khai báo", ha="center", color=MUTED, fontsize=9)
ax.set_title("Ma trận thuộc tính: đậm = mạnh nhất trong cột\n", loc="left")
plt.tight_layout()
save(fig, "3-trait-heatmap")

In [ ]:
dots = np.ceil(norm_traits * 3).astype("Int64").rename(index=NAME)
dots = dots.apply(lambda col: col.map(lambda v: "" if pd.isna(v) or v == 0 else "●" * int(v)))
save_table(dots, "3-trait-dots")

## 4. Lưới ảnh mẫu

Trang đầu mỗi loại, bản sạch, khung trường tô theo kiểu điền. Đổi `GALLERY_DOSSIER` để xem hồ sơ khác (`None` = hồ sơ đầu tiên có loại đó).

In [ ]:
GALLERY_DOSSIER = None
GALLERY_PAGE = 1

def page_row(doc_type, level="clean", page=GALLERY_PAGE, dossier=GALLERY_DOSSIER):
    d = pages[(pages.doc_type == doc_type) & (pages.level == level) & (pages.page == page)]
    if dossier is not None:
        d = d[d.dossier == dossier]
    return d.iloc[0] if len(d) else None

def show_page(ax, row, boxes=True, max_side=1400):
    img = cv2.cvtColor(cv2.imread(row.path), cv2.COLOR_BGR2RGB)
    k = min(1.0, max_side / max(img.shape[:2]))
    ax.imshow(cv2.resize(img, None, fx=k, fy=k, interpolation=cv2.INTER_AREA))
    if boxes and row.level == "clean":
        f = fields[(fields.dossier == row.dossier) & (fields.doc_id == row.doc_id)]
        for fill, bb in ((r.fill, b) for r in f.itertuples() for b in r.bbox if b.get("page", 1) == row.page):
            ax.add_patch(plt.Rectangle((bb["x"] * k, bb["y"] * k), bb["w"] * k, bb["h"] * k, fill=False,
                                       edgecolor=FILL_COLOR.get(fill, MUTED), linewidth=1.2))
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_color(GRID)

rows = [r for r in (page_row(t) for t in TYPE_ORDER) if r is not None]
ncol = 4
nrow = int(np.ceil(len(rows) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(4.2 * ncol, 5.2 * nrow))
for ax in axes.flat:
    ax.axis("off")
for ax, r in zip(axes.flat, rows):
    ax.axis("on")
    show_page(ax, r)
    d = docs[(docs.dossier == r.dossier) & (docs.doc_id == r.doc_id)].iloc[0]
    ax.set_title(f"{NAME[r.doc_type]}\n{d.variant} · {d.n_pages} trang · {d.page_format}/{d.paper} · cũ {d.age:.2f}",
                 loc="left", fontsize=9)
handles = [plt.Rectangle((0, 0), 1, 1, fill=False, edgecolor=c, linewidth=2) for c in FILL_COLOR.values()]
fig.legend(handles, FILL_NAME.values(), frameon=False, ncol=3, loc="lower center", bbox_to_anchor=(0.5, -0.01))
plt.tight_layout(rect=(0, 0.02, 1, 1))
save(fig, "4-gallery")

Cùng một trang qua các mức nhiễu. Đổi `NOISE_DOC_TYPE` để xem loại khác. Phép biến đổi đã áp ghi dưới mỗi ảnh.

In [ ]:
NOISE_DOC_TYPE = "giay-chung-nhan-nha-dat" if "giay-chung-nhan-nha-dat" in TYPE_ORDER else TYPE_ORDER[0]

base = page_row(NOISE_DOC_TYPE)
same = pages[(pages.dossier == base.dossier) & (pages.doc_id == base.doc_id) & (pages.page == base.page)].sort_values("level")
fig, axes = plt.subplots(1, len(same), figsize=(4.2 * len(same), 5.6))
for ax, r in zip(np.atleast_1d(axes), same.itertuples()):
    show_page(ax, r, boxes=False)
    ax.set_title(r.level, loc="left", fontsize=10)
    ax.set_xlabel("\n".join(r.ops) or "ảnh gốc", fontsize=8, color=MUTED)
fig.suptitle(f"{NAME[NOISE_DOC_TYPE]}: trang {base.page} theo mức nhiễu", x=0.01, ha="left")
plt.tight_layout()
save(fig, "4-noise-levels")

## 5. Đặc trưng ảnh, văn bản, bố cục

### 5.1. Ảnh

Đo trên ảnh xám, sau khi thu nhỏ cạnh dài về `IMAGE_MAX_SIDE`:

| Số đo | Cách tính | Ý nghĩa |
|---|---|---|
| Độ sáng | Trung bình mức xám | Giấy ố, bóng tối làm giảm |
| Độ tương phản | Độ lệch chuẩn mức xám | Mực phai, giấy ngả màu làm giảm |
| Độ nét | Phương sai Laplacian (log10) | Mờ làm giảm |
| Nhiễu | Ước lượng σ theo Immerkær (1996) | Hạt, JPEG làm tăng |
| Tỷ lệ mực | Tỷ lệ điểm ảnh tối hơn ngưỡng Otsu | Mật độ chữ và nét vẽ |
| Độ nghiêng | `render.tilt` + phép `rotate` của generator (độ) | Lấy giá trị thật, không ước lượng |

Kết quả cache ở `outputs/dataset-profile/image_stats.csv`, chạy lại chỉ đo ảnh mới. Lần đầu mất khoảng 1 phút mỗi 1000 ảnh.

In [ ]:
IMMERKAER = np.array([[1, -2, 1], [-2, 4, -2], [1, -2, 1]], dtype=np.float32)

def image_stats(path):
    gray = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    h, w = gray.shape
    k = min(1.0, IMAGE_MAX_SIDE / max(h, w))
    g = cv2.resize(gray, None, fx=k, fy=k, interpolation=cv2.INTER_AREA) if k < 1 else gray
    gf = g.astype(np.float32)
    otsu, _ = cv2.threshold(g, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    noise = np.sqrt(np.pi / 2) * np.abs(cv2.filter2D(gf, -1, IMMERKAER)).sum() / (6 * (g.shape[0] - 2) * (g.shape[1] - 2))
    return dict(path=path, width=w, height=h, brightness=gf.mean(), contrast=gf.std(),
                sharpness=np.log10(cv2.Laplacian(gf, cv2.CV_32F).var() + 1e-6), noise=noise, ink=(g < otsu).mean())

stats_path = FIG_DIR / "image_stats.csv"
cached = pd.read_csv(stats_path) if stats_path.exists() else pd.DataFrame(columns=["path"])
todo = sorted(set(pages.path) - set(cached.path))
if todo:
    print(f"Đo {len(todo)} ảnh ({len(cached)} đã cache)")
    cached = pd.concat([cached, pd.DataFrame(map(image_stats, todo))], ignore_index=True)
    cached.to_csv(stats_path, index=False)
img = pages.merge(cached, on="path")
dims = docs.set_index(["dossier", "doc_id"])[["w_mm", "dpi", "page_format"]]
img = img.join(dims, on=["dossier", "doc_id"])
img["dpi_eff"] = img.width / (img.w_mm / 25.4)      # DPI thực của ảnh so với khổ giấy khai báo
img.head(3)

Độ phân giải theo loại (ảnh sạch). Khổ thẻ nhỏ nên ít điểm ảnh dù cùng DPI.

In [ ]:
res = img[img.level == "clean"].assign(mp=lambda d: d.width * d.height / 1e6).groupby("doc_type", observed=True).agg(
    **{"Khổ": ("page_format", lambda s: s.mode().iat[0]),
       "Rộng (px)": ("width", "median"), "Cao (px)": ("height", "median"), "Megapixel": ("mp", "median"),
       "DPI khai báo": ("dpi", "median"), "DPI thực": ("dpi_eff", "median")})
save_table(res.rename(index=NAME), "5-resolution").style.format(precision=1)

In [ ]:
IMAGE_METRICS = [("contrast", "Độ tương phản (σ mức xám)"), ("sharpness", "Độ nét (log10 var Laplacian)"),
                 ("noise", "Nhiễu (σ Immerkær)"), ("skew", "Độ nghiêng (độ)")]

def grouped_box(ax, df, col):
    """Mỗi loại một cụm, mỗi mức nhiễu một hộp trong cụm."""
    width = 0.8 / len(LEVELS)
    for j, lv in enumerate(LEVELS):
        data = [df[(df.doc_type == t) & (df.level == lv)][col].dropna().values for t in TYPE_ORDER]
        pos = np.arange(len(TYPE_ORDER)) + (j - (len(LEVELS) - 1) / 2) * width
        keep = [i for i, d in enumerate(data) if len(d)]
        if not keep:
            continue
        bp = ax.boxplot([data[i] for i in keep], positions=pos[keep], widths=width * 0.85, patch_artist=True,
                        showfliers=False, medianprops=dict(color=SURFACE, linewidth=1.5),
                        whiskerprops=dict(color=LEVEL_COLOR[lv]), capprops=dict(color=LEVEL_COLOR[lv]))
        for b in bp["boxes"]:
            b.set(facecolor=LEVEL_COLOR[lv], edgecolor=LEVEL_COLOR[lv])
    ax.set_xticks(range(len(TYPE_ORDER)), [NAME[t] for t in TYPE_ORDER], rotation=20, ha="right")
    style_axes(ax)

fig, axes = plt.subplots(2, 2, figsize=(16, 9))
for ax, (col, title) in zip(axes.flat, IMAGE_METRICS):
    grouped_box(ax, img, col)
    ax.set_title(title, loc="left")
handles = [plt.Rectangle((0, 0), 1, 1, color=LEVEL_COLOR[lv]) for lv in LEVELS]
fig.legend(handles, LEVELS, frameon=False, ncol=4, loc="lower center", bbox_to_anchor=(0.5, -0.02))
plt.tight_layout(rect=(0, 0.03, 1, 1))
save(fig, "5-image-stats")

In [ ]:
(img.pivot_table(index="doc_type", columns="level", values=["brightness", "contrast", "sharpness", "noise", "ink"],
                 aggfunc="median", observed=True)
    .rename(index=NAME).style.format("{:.2f}"))

### 5.2. Văn bản

Tính trên văn bản đầy đủ của từng trang sạch (`files.text`, gồm cả chữ in sẵn của mẫu), trừ tỷ lệ kiểu điền tính trên ký tự của trường có nhãn.

- **Tỷ lệ ký tự có dấu:** trong các chữ cái, tỷ lệ chữ mang dấu tiếng Việt (kể cả `đ`). Cao thì model yếu về dấu sẽ thiệt nhiều.
- **Từ vựng:** số từ khác nhau (viết thường) mỗi loại, và tỷ lệ từ khác nhau / tổng số từ (TTR). TTR thấp là văn bản lặp mẫu nhiều.

In [ ]:
def has_mark(c):
    return c in "đĐ" or any(unicodedata.category(x) == "Mn" for x in unicodedata.normalize("NFD", c))

def text_stats(text):
    words = re.findall(r"\w+", text)
    letters = [c for c in text if c.isalpha()]
    return dict(chars=len(text.replace("\n", "")), words=len(words), lines=sum(bool(l.strip()) for l in text.splitlines()),
                marked=sum(map(has_mark, letters)) / max(len(letters), 1),
                upper=sum(c.isupper() for c in letters) / max(len(letters), 1),
                word_len=np.mean([len(w) for w in words]) if words else np.nan)

txt = clean.join(pd.DataFrame(map(text_stats, clean.text), index=clean.index))
vocab = txt.groupby("doc_type", observed=True).text.apply(lambda s: re.findall(r"\w+", " ".join(s).lower()))
text_table = pd.DataFrame({
    "Ký tự/trang": txt.groupby("doc_type", observed=True).chars.median(),
    "Từ/trang": txt.groupby("doc_type", observed=True).words.median(),
    "Dòng/trang": txt.groupby("doc_type", observed=True).lines.median(),
    "Độ dài từ TB": txt.groupby("doc_type", observed=True).word_len.mean(),
    "% chữ có dấu": txt.groupby("doc_type", observed=True).marked.mean(),
    "% chữ hoa": txt.groupby("doc_type", observed=True).upper.mean(),
    "Từ khác nhau": vocab.map(lambda w: len(set(w))),
    "TTR": vocab.map(lambda w: len(set(w)) / max(len(w), 1)),
}).join(share.rename(columns=lambda c: f"% ký tự {FILL_NAME.get(c, c).lower()}"))
save_table(text_table.rename(index=NAME), "5-text-stats")
(text_table.rename(index=NAME).style
   .format("{:,.0f}").format("{:.1f}", subset=["Độ dài từ TB"])
   .format("{:.1%}", subset=[c for c in text_table.columns if c.startswith("%") or c == "TTR"]))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 0.5 * len(TYPE_ORDER) + 1.6), gridspec_kw=dict(width_ratios=[1.3, 1, 1]))
# Kiểu điền: 100% stacked, khe 2px giữa các đoạn
ax, y, left = axes[0], np.arange(len(TYPE_ORDER)), np.zeros(len(TYPE_ORDER))
for fill in FILL_NAME:
    v = share.reindex(TYPE_ORDER).get(fill, pd.Series(0.0, index=TYPE_ORDER)).fillna(0).values
    ax.barh(y, v, left=left, height=0.62, color=FILL_COLOR[fill], edgecolor=SURFACE, linewidth=2, label=FILL_NAME[fill])
    for yi, l, w in zip(y, left, v):
        if w >= 0.08:
            ax.text(l + w / 2, yi, f"{w:.0%}", ha="center", va="center", fontsize=8, color="white" if fill != "printed" else INK)
    left += v
ax.set_yticks(y, [NAME[t] for t in TYPE_ORDER])
ax.invert_yaxis()
ax.set_xlim(0, 1)
ax.set_title("Ký tự có nhãn theo kiểu điền", loc="left")
style_axes(ax, grid="x", percent_axis="x")
ax.legend(frameon=False, ncol=3, loc="lower left", bbox_to_anchor=(0, 1.06), fontsize=8)

ax = axes[1]
data = [txt[txt.doc_type == t].chars.values for t in TYPE_ORDER]
bp = ax.boxplot(data, vert=False, widths=0.6, patch_artist=True, showfliers=False, medianprops=dict(color=SURFACE, linewidth=1.5))
for b, t in zip(bp["boxes"], TYPE_ORDER):
    b.set(facecolor=COLOR[t], edgecolor=COLOR[t])
ax.set_yticks(range(1, len(TYPE_ORDER) + 1), [])
ax.invert_yaxis()
ax.set_title("Ký tự mỗi trang", loc="left")
style_axes(ax, grid="x")

type_barh(axes[2], text_table["% chữ có dấu"].reindex(TYPE_ORDER), fmt="{:.0%}")
axes[2].set_yticklabels([])
axes[2].set_title("Tỷ lệ chữ có dấu", loc="left")
axes[2].xaxis.set_major_formatter(pct)
plt.tight_layout()
save(fig, "5-text")

Kiểu dữ liệu của trường theo loại giấy tờ: số trường trung bình mỗi giấy tờ. Cho biết tầng trích xuất (AIP-005) phải xử lý kiểu nào ở đâu.

In [ ]:
ftype = (fields.groupby(["doc_type", "field_type"], observed=True).size()
         .div(by_type(docs).size(), level="doc_type").unstack(fill_value=0))
ftype = ftype[ftype.sum().sort_values(ascending=False).index]
save_table(ftype.rename(index=NAME), "5-field-types")
ftype.rename(index=NAME).style.format("{:.1f}").background_gradient(cmap=SEQ, axis=None)

### 5.3. Bố cục

Ground truth có bbox cho trường, chưa có nhãn vùng bảng/hình/dấu, nên bố cục đo qua trường:

- **Số dòng chữ có nhãn mỗi trang** và **mật độ**: tổng diện tích bbox / diện tích trang.
- **Dấu vân bố cục:** mật độ vị trí tâm bbox trên trang 1 (tọa độ chuẩn hóa 0–1), cộng dồn mọi giấy tờ cùng loại. Loại có mẫu cố định cho các cụm sắc nét; loại bố cục tự do (giấy sang đất) cho vệt nhòe.

In [ ]:
pages_px = img[img.level == "clean"][["dossier", "doc_id", "page", "width", "height"]]
boxes = pd.DataFrame([dict(dossier=r.dossier, doc_id=r.doc_id, doc_type=r.doc_type, fill=r.fill, page=b.get("page", 1),
                           x=b["x"], y=b["y"], w=b["w"], h=b["h"])
                      for r in fields.itertuples() for b in r.bbox])
boxes = boxes.merge(pages_px, on=["dossier", "doc_id", "page"]).assign(area=lambda d: d.w * d.h)
boxes["doc_type"] = pd.Categorical(boxes["doc_type"], TYPE_ORDER, ordered=True)
per_page = boxes.groupby(["doc_type", "dossier", "doc_id", "page"], observed=True).agg(
    n=("x", "size"), area=("area", "sum"), W=("width", "first"), H=("height", "first"))
per_page["density"] = per_page.area / (per_page.W * per_page.H)
layout = per_page.groupby("doc_type", observed=True).agg(
    **{"Dòng chữ/trang (TB)": ("n", "mean"), "Dòng chữ/trang (max)": ("n", "max"), "Mật độ bbox": ("density", "mean")})
save_table(layout.rename(index=NAME), "5-layout")
layout.rename(index=NAME).style.format("{:.1f}").format("{:.1%}", subset=["Mật độ bbox"])

In [ ]:
BINS = 40
p1 = boxes[boxes.page == 1].assign(cx=lambda d: (d.x + d.w / 2) / d.width, cy=lambda d: (d.y + d.h / 2) / d.height)
ncol = 4
nrow = int(np.ceil(len(TYPE_ORDER) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(3.6 * ncol, 4.4 * nrow))
for ax in axes.flat:
    ax.axis("off")
for ax, t in zip(axes.flat, TYPE_ORDER):
    d = p1[p1.doc_type == t]
    H, _, _ = np.histogram2d(d.cy, d.cx, bins=BINS, range=[[0, 1], [0, 1]])
    ax.imshow(np.log1p(H), cmap=SEQ, extent=(0, 1, 1, 0), aspect=float(d.height.median() / d.width.median()) if len(d) else 1)
    ax.set_title(f"{NAME[t]} ({d.groupby(['dossier', 'doc_id']).ngroups} giấy tờ)", loc="left", fontsize=10)
    ax.axis("on"); ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_color(GRID)
fig.suptitle("Dấu vân bố cục: vị trí tâm bbox trường, trang 1 (đậm = nhiều)", x=0.01, ha="left")
plt.tight_layout()
save(fig, "5-layout-fingerprint")

## 6. Lược đồ nhãn

`ground_truth.json` có 4 cấp nhãn, sinh cùng lúc với ảnh nên không phải gán tay. JSON Schema đầy đủ ở `src/data-golden-set-generator/specs/ground_truth.schema.json`.

```mermaid
flowchart TD
    D[Hồ sơ<br/>decision.label, reasons, compensation, review] --> P[Người, thửa đất, dòng thời gian<br/>persona, people, property, timeline]
    D --> G[Giấy tờ<br/>doc_type, variant, issue_date, render]
    G --> PG[Trang<br/>ảnh sạch + ảnh nhiễu, ops, text]
    G --> F[Trường<br/>name, type, value, text, fill, mutated_by]
    F --> B[Bbox mỗi dòng<br/>page, x, y, w, h; biến đổi theo từng ảnh nhiễu]
```

| Cấp | Nhãn | Dùng cho |
|---|---|---|
| Hồ sơ | `decision.label` (approve / request_supplement / reject), `reasons` (quy tắc, ca lỗi), `missing_documents` | Đánh giá cuối (AIP-027), quyết định 3 hướng (AIP-020) |
| Giấy tờ | `doc_type`, `variant`, `issue_date`, `render` | Phân loại, tách giấy tờ (AIP-004) |
| Trang | Đường dẫn ảnh sạch/nhiễu, `ops`, `text` cả trang | OCR cả trang (AIP-001), kiểm tra chất lượng ảnh (AIP-003) |
| Trường | `name`, `type`, `value` (chuẩn hóa), `text` (in trên giấy), `fill`, `mutated_by`, `bbox` | Trích xuất (AIP-005), grounding (AIP-006), OCR theo trường |

Ví dụ một bản ghi mỗi cấp, lấy từ hồ sơ đầu tiên:

In [ ]:
gt = json.loads(next(iter(sorted(DATA_DIR.rglob("ground_truth.json")))).read_text(encoding="utf-8"))
doc = next((d for d in gt["documents"] if any(f.get("fill") == "handwritten" for f in d["fields"])), gt["documents"][0])

def short(o, n=90):
    s = json.dumps(o, ensure_ascii=False)
    return s if len(s) <= n else s[:n] + "…"

example = {
    "Hồ sơ": {k: gt[k] for k in ("dossier_id", "procedure", "submit_date", "decision") if k in gt},
    "Giấy tờ": {k: doc[k] for k in ("doc_id", "doc_type", "variant", "issue_date", "render")},
    "Trang (nhiễu)": {k: v for k, v in (doc["files"].get("noisy") or [{}])[0].items() if k != "bboxes"},
    "Trường": next((f for f in doc["fields"] if f.get("fill") == "handwritten"), doc["fields"][0]),
}
for level, rec in example.items():
    print(f"── {level}")
    for k, v in rec.items():
        print(f"   {k:<12} {short(v)}")

Danh mục trường mỗi loại: nhãn tiếng Việt lấy từ `schema.yaml` (nếu có `TEMPLATES_DIR`), kiểu điền thực tế trong dữ liệu (một trường có thể đổi kiểu điền theo biến thể), tỷ lệ giấy tờ có trường đó khác rỗng và tỷ lệ bị gây lỗi có chủ đích.

In [ ]:
labels = {}
try:
    import yaml
    for p in TEMPLATES_DIR.glob("*/*/schema.yaml"):
        s = yaml.safe_load(p.read_text(encoding="utf-8"))
        labels.update({(s["doc_type"], f["name"]): f.get("label", "") for f in s.get("fields", [])})
except ImportError:
    print("Không có PyYAML, bỏ cột nhãn")

n_docs = by_type(docs).size()
inventory = (fields.groupby(["doc_type", "field"], observed=True)
             .agg(type=("field_type", "first"), fill=("fill", lambda s: " / ".join(sorted(s.unique()))),
                  filled=("chars", lambda s: (s > 0).sum()), mutated=("mutated", "mean"), lines=("n_lines", "mean")))
inventory["label"] = [labels.get(k, "") for k in inventory.index]
inventory["filled"] = inventory.filled / n_docs.reindex(inventory.index.get_level_values(0)).values
inventory = inventory[["label", "type", "fill", "filled", "lines", "mutated"]]
save_table(inventory, "6-field-inventory")
inventory.style.format({"filled": "{:.0%}", "mutated": "{:.1%}", "lines": "{:.1f}"})

## 7. Không gian đặc trưng

Trích embedding ảnh trang bằng backbone `EMBED_MODEL` (ResNet trọng số ImageNet, bỏ lớp phân loại), chiếu xuống 2 chiều bằng t-SNE, tô màu theo loại. Trang được đặt vào khung vuông nền trắng giữ nguyên tỷ lệ (không cắt), vì khổ thẻ, A4, khổ sổ khác nhau.

Hình chiếu t-SNE chỉ để nhìn. Để đo chồng lấn, dùng **ma trận láng giềng**: với mỗi trang, lấy `K` trang gần nhất (cosine, trên embedding gốc), bỏ các trang của chính giấy tờ đó, đếm láng giềng thuộc loại nào. Đường chéo cao là loại tách rời; ô ngoài đường chéo cao là hai loại dễ nhầm khi phân loại (AIP-004).

Lần đầu tải trọng số (~45MB) và trích đặc trưng (CPU khoảng vài phút mỗi 1000 ảnh). Cache ở `outputs/dataset-profile/embed_<model>.npz`.

In [ ]:
import torch
from PIL import Image
from torchvision.models import get_model, get_model_weights

K = 5
emb_pages = pages[pages.level.isin(EMBED_LEVELS)].reset_index(drop=True)
emb_path = FIG_DIR / f"embed_{EMBED_MODEL}.npz"
cache = dict(np.load(emb_path, allow_pickle=True)) if emb_path.exists() else {"paths": np.array([]), "feats": np.zeros((0, 0))}
known = dict(zip(cache["paths"], cache["feats"]))
todo = [p for p in emb_pages.path if p not in known]

def letterbox(path, size=224):
    im = Image.open(path).convert("RGB")
    im.thumbnail((size, size), Image.Resampling.LANCZOS)
    canvas = Image.new("RGB", (size, size), "white")
    canvas.paste(im, ((size - im.width) // 2, (size - im.height) // 2))
    x = torch.from_numpy(np.asarray(canvas, dtype=np.float32) / 255).permute(2, 0, 1)
    return (x - torch.tensor([0.485, 0.456, 0.406])[:, None, None]) / torch.tensor([0.229, 0.224, 0.225])[:, None, None]

if todo:
    print(f"Trích đặc trưng {len(todo)} ảnh ({len(known)} đã cache)")
    net = get_model(EMBED_MODEL, weights=get_model_weights(EMBED_MODEL).DEFAULT).eval()
    net.fc = torch.nn.Identity()
    with torch.no_grad():
        for i in range(0, len(todo), 32):
            batch = todo[i:i + 32]
            known.update(zip(batch, net(torch.stack([letterbox(p) for p in batch])).numpy()))
    np.savez(emb_path, paths=np.array(list(known)), feats=np.stack(list(known.values())))

X = np.stack([known[p] for p in emb_pages.path])
X = X / np.linalg.norm(X, axis=1, keepdims=True)
X.shape

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

Z = PCA(n_components=min(50, *X.shape), random_state=0).fit_transform(X)
Y = TSNE(n_components=2, perplexity=min(30, max(5, len(X) // 10)), init="pca", random_state=0).fit_transform(Z)

fig, axes = plt.subplots(1, len(EMBED_LEVELS), figsize=(7.5 * len(EMBED_LEVELS), 6.5), squeeze=False)
for ax, lv in zip(axes[0], EMBED_LEVELS):
    for t in TYPE_ORDER:
        m = ((emb_pages.doc_type == t) & (emb_pages.level == lv)).values
        ax.scatter(Y[m, 0], Y[m, 1], s=34, color=COLOR[t], marker=MARKER[t], edgecolor=SURFACE, linewidth=0.8, label=NAME[t])
    for t in TYPE_ORDER:   # nhãn trực tiếp tại tâm cụm
        m = ((emb_pages.doc_type == t) & (emb_pages.level == lv)).values
        if m.any():
            ax.text(*np.median(Y[m], axis=0), NAME[t], fontsize=8, color=INK, ha="center", va="bottom",
                    bbox=dict(boxstyle="round,pad=0.2", facecolor=SURFACE, edgecolor="none", alpha=0.8))
    ax.set_title(f"t-SNE {EMBED_MODEL}, ảnh {lv}", loc="left")
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_color(GRID)
axes[0, -1].legend(frameon=False, fontsize=8, loc="upper left", bbox_to_anchor=(1.01, 1))
plt.tight_layout()
save(fig, "7-tsne")

In [ ]:
sim = X @ X.T
same_doc = (emb_pages.dossier.values[:, None] == emb_pages.dossier.values) & (emb_pages.doc_id.values[:, None] == emb_pages.doc_id.values)
sim[same_doc] = -np.inf                                   # bỏ chính nó và các trang/ảnh nhiễu của cùng giấy tờ
nn = np.argsort(-sim, axis=1)[:, :K]
types = emb_pages.doc_type.astype(str).values
neigh = pd.crosstab(np.repeat(types, K), types[nn].ravel(), normalize="index").reindex(index=TYPE_ORDER, columns=TYPE_ORDER).fillna(0)
save_table(neigh.rename(index=NAME, columns=NAME), "7-knn-confusion")

fig, ax = plt.subplots(figsize=(0.9 * len(TYPE_ORDER) + 3, 0.7 * len(TYPE_ORDER) + 1.5))
ax.imshow(neigh.values, cmap=SEQ, vmin=0, vmax=1)
for i in range(len(TYPE_ORDER)):
    for j in range(len(TYPE_ORDER)):
        v = neigh.values[i, j]
        if v >= 0.005:
            ax.text(j, i, f"{v:.0%}", ha="center", va="center", fontsize=9, color="white" if v > 0.55 else INK)
ax.set_xticks(range(len(TYPE_ORDER)), [NAME[t] for t in TYPE_ORDER], rotation=30, ha="right")
ax.set_yticks(range(len(TYPE_ORDER)), [NAME[t] for t in TYPE_ORDER])
ax.set_xlabel(f"Loại của {K} láng giềng gần nhất")
ax.tick_params(length=0)
for s in ax.spines.values():
    s.set_visible(False)
ax.set_title(f"Ma trận láng giềng ({', '.join(EMBED_LEVELS)}): đường chéo = tách rời", loc="left")
plt.tight_layout()
save(fig, "7-knn-confusion")

## 8. Kết quả benchmark theo loại

Đọc điểm từng trường mà `ocr-eval.ipynb` ghi ra (`outputs/scores_e2e.csv`, `outputs/scores_rec.csv`), không chạy OCR lại. `ocr-eval.ipynb` mặc định `MAX_IMAGES = 24` nên chỉ phủ vài loại giấy tờ; đặt `MAX_IMAGES = None` và chạy lại để có đủ 7 loại. Ô có ít hơn `MIN_FIELDS` trường để trống.

In [ ]:
scores = {n: pd.read_csv(OUTPUT_DIR / f"scores_{n}.csv") for n in ("e2e", "rec") if (OUTPUT_DIR / f"scores_{n}.csv").exists()}
if not scores:
    print("Chưa có outputs/scores_*.csv: chạy ocr-eval.ipynb trước. Bỏ qua phần 8.")
for df in scores.values():
    df["doc_type"] = pd.Categorical(df["doc_type"], TYPE_ORDER, ordered=True)
MODELS = list(dict.fromkeys(next(iter(scores.values())).model)) if scores else []

def per_type(df, value, by="model"):
    g = df.groupby(["doc_type", by], observed=True)[value]
    return g.mean().where(g.size() >= MIN_FIELDS).unstack(by)

pd.DataFrame({f"Số trường ({n})": df.groupby("doc_type", observed=True).size() for n, df in scores.items()})

CER theo loại × model. Hàng đậm là loại khó với mọi model; cột đậm là model yếu.

In [ ]:
if scores:
    fig, axes = plt.subplots(1, len(scores), figsize=(1.1 * len(MODELS) * len(scores) + 4, 0.55 * len(TYPE_ORDER) + 2.5),
                             sharey=True, squeeze=False)
    for ax, (n, df) in zip(axes[0], scores.items()):
        t = per_type(df, "cer").reindex(index=TYPE_ORDER, columns=MODELS)
        save_table(t.rename(index=NAME), f"8-cer-{n}")
        ax.imshow(t.values.astype(float), cmap=SEQ, vmin=0, vmax=0.5, aspect="auto")
        for i in range(t.shape[0]):
            for j in range(t.shape[1]):
                v = t.values[i, j]
                ax.text(j, i, "—" if pd.isna(v) else f"{v:.1%}", ha="center", va="center", fontsize=8,
                        color=MUTED if pd.isna(v) else ("white" if v > 0.28 else INK))
        ax.set_xticks(range(len(MODELS)), MODELS, rotation=35, ha="right", fontsize=8)
        ax.set_yticks(range(len(TYPE_ORDER)), [NAME[t] for t in TYPE_ORDER])
        ax.tick_params(length=0)
        for s in ax.spines.values():
            s.set_visible(False)
        ax.set_title(f"CER {'end-to-end' if n == 'e2e' else 'recognition-only'} (thấp là tốt)", loc="left")
    plt.tight_layout()
    save(fig, "8-cer-by-type")

Field Accuracy theo loại, dạng radar: mỗi trục một loại giấy tờ, mỗi đường một model (màu theo engine, nét theo model trong engine, giống `ocr-eval.ipynb`). Đa giác lõm vào ở trục nào là loại đó kéo độ chính xác xuống.

In [ ]:
ENGINE_COLOR = {"paddleocr": "#2a78d6", "vietocr": "#eb6834", "tesseract": "#1baf7a"}
DASHES = ["-", "--", ":"]

if scores:
    df = scores.get("rec", next(iter(scores.values())))
    acc = per_type(df.assign(exact=df.cer == 0), "exact").reindex(index=TYPE_ORDER, columns=MODELS)
    axes_types = [t for t in TYPE_ORDER if acc.loc[t].notna().any()]
    if len(axes_types) < 3:
        print(f"Chỉ có {len(axes_types)} loại đủ {MIN_FIELDS} trường, radar cần ít nhất 3: chạy ocr-eval với MAX_IMAGES = None")
    else:
        angles = np.linspace(0, 2 * np.pi, len(axes_types), endpoint=False)
        fig, ax = plt.subplots(figsize=(8, 7), subplot_kw=dict(polar=True))
        for m in MODELS:
            engine = m.split("/")[0]
            same = [x for x in MODELS if x.split("/")[0] == engine]
            v = acc.loc[axes_types, m].fillna(0).values
            ax.plot(np.r_[angles, angles[0]], np.r_[v, v[0]], color=ENGINE_COLOR.get(engine, MUTED),
                    linestyle=DASHES[same.index(m) % 3], linewidth=2, marker="o", markersize=5, label=m)
        ax.set_xticks(angles, [NAME[t] for t in axes_types], fontsize=9)
        ax.set_ylim(0, 1)
        ax.yaxis.set_major_formatter(pct)
        ax.tick_params(colors=MUTED)
        ax.grid(color=GRID)
        ax.spines["polar"].set_color(GRID)
        ax.set_title(f"Field Accuracy theo loại ({'rec-only' if 'rec' in scores else 'end-to-end'})", loc="left")
        ax.legend(frameon=False, fontsize=8, loc="upper left", bbox_to_anchor=(1.05, 1))
        plt.tight_layout()
        save(fig, "8-radar")

### Đặc trưng nào làm OCR khó hơn

Mỗi điểm là một loại giấy tờ: trục ngang là một trục của ma trận thuộc tính (phần 3), trục dọc là CER trung bình mọi model. Với 7 điểm chỉ nhìn được xu hướng, không đủ để kết luận thống kê; bảng dưới tách CER theo kiểu điền × loại để thấy phần khó nằm ở chữ viết tay hay ở loại giấy tờ.

In [ ]:
TRAIT_AXES = ["Viết tay", "Giấy cũ", "Mật độ trường", "Biến thể mẫu"]

if scores:
    df = scores.get("rec", next(iter(scores.values())))
    cer_type = df.groupby("doc_type", observed=True).cer.mean()
    fig, axes = plt.subplots(1, len(TRAIT_AXES), figsize=(4.2 * len(TRAIT_AXES), 3.8), sharey=True)
    for ax, c in zip(axes, TRAIT_AXES):
        for t in cer_type.index:
            ax.scatter(traits.loc[t, c], cer_type[t], s=60, color=COLOR[t], marker=MARKER[t], edgecolor=SURFACE, linewidth=1)
            ax.annotate(NAME[t], (traits.loc[t, c], cer_type[t]), xytext=(5, 3), textcoords="offset points", fontsize=8)
        ax.set_xlabel(c)
        style_axes(ax, grid="both", percent_axis="y")
    axes[0].set_ylabel("CER trung bình")
    fig.suptitle("Thuộc tính của loại giấy tờ và CER", x=0.01, ha="left")
    plt.tight_layout()
    save(fig, "8-traits-vs-cer")

    by_fill = df.pivot_table(index="doc_type", columns="fill", values="cer", aggfunc="mean", observed=True)
    display(by_fill.rename(index=NAME, columns=FILL_NAME).style.format("{:.1%}", na_rep="—")
            .background_gradient(cmap=SEQ, vmin=0, vmax=0.5))

## Ghi chú đọc kết quả

- **Mọi số đo là của dữ liệu tổng hợp.** Chữ viết tay là font viết tay, nhiễu là Augraphy; phân bố thật (ảnh chụp điện thoại, chữ viết tay người thật, dấu đỏ đè chữ) rộng hơn. Khi có `real/`, chạy lại notebook trên đó và đặt hai bộ số cạnh nhau để thấy khoảng cách.
- **Phân bố lớp đều là do thiết kế:** mỗi hồ sơ luôn đủ 7 giấy tờ và kịch bản xoay vòng. Hồ sơ thật lệch hơn nhiều (thiếu giấy tờ, nhiều bản sao), nên metric gộp trên golden set phải tính theo loại rồi mới gộp.
- **Trục khai báo tay** (`DECLARED_TRAITS`) không tự cập nhật, cần sửa khi thêm loại giấy tờ hoặc đổi template.
- **Ma trận láng giềng** dùng backbone ImageNet, nhìn hình dạng trang chứ không đọc chữ. Hai loại chồng lấn ở đây nhưng khác nội dung vẫn tách được bằng OCR + từ khóa; chồng lấn ở cả hai mới là ca khó cho phân loại.